In [ ]:
!pip install pandas openpyxl matplotlib numpy

In [ ]:
#================================================================
# FIGURE 1 AND FIGURE 2
#................................................................
# Figure 1: Description
# Concept-level Candidate-Set Size Boxplot
#................................................................
# Figure 2: Description
# Concept-level F1 Boxplot
#................................................................
# Evaluation basis:
# NC = 2,441 concepts
# NT = 10,161 valid query variants per algorithm
# mc = 2 to 6 available spelling variants per concept
# Empty Query cells are excluded completely to avoid result pollution
# Valid observations are those which are genuine non-empty queries 
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# CONFIGURATION
# ============================================================+++++

INPUT_FILE = (
    r"D:\BORUE"
    r"\PhoneticEvaluation.xlsx"
)

OUTPUT_DIR = Path(
    r"D:\BORUE\PaperFigures10Nov"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# EXPECTED DATASET VALUES
# ============================================================

EXPECTED_PHYSICAL_ROWS = 146460
EXPECTED_BLANK_ROWS = 44850
EXPECTED_VALID_ROWS = 101610

EXPECTED_ALGORITHMS = 10
EXPECTED_CONCEPTS = 2441
EXPECTED_VALID_VARIANTS_PER_ALGORITHM = 10161


# ============================================================
# ALGORITHM ORDER
# ============================================================

ALGORITHM_ORDER = [
    "Soundex",
    "Refined Soundex",
    "Cologne",
    "NYSIIS",
    "Metaphone",
    "Double Metaphone",
    "Caverphone1",
    "Caverphone2",
    "UrduPhone",
    "RomanMetaphone"
]


# ============================================================
# DISPLAY NAMES
# ============================================================

DISPLAY_NAMES = {
    "Soundex": "Soundex",
    "Refined Soundex": "Refined\nSoundex",
    "Cologne": "Cologne",
    "NYSIIS": "NYSIIS",
    "Metaphone": "Metaphone",
    "Double Metaphone": "Double\nMetaphone",
    "Caverphone1": "Caverphone1",
    "Caverphone2": "Caverphone2",
    "UrduPhone": "UrduPhone",
    "RomanMetaphone": "RASOM"
}


# ============================================================
# LOAD DATA
# ============================================================

print("Reading:", INPUT_FILE)

df = pd.read_excel(
    INPUT_FILE,
    engine="openpyxl",
    usecols=[
        "Algorithm",
        "ConceptRow",
        "VariantNo",
        "Query",
        "RetrievedCount",
        "F1"
    ]
)

print()
print("Physical rows read:", len(df))


# ============================================================
# REMOVE BLANK-QUERY PLACEHOLDERS
# ============================================================
#
# Empty Query:
#     unavailable spelling variant
#     -> exclude
#
# Non-empty Query with RetrievedCount = 0:
#     genuine failed retrieval
#     -> retain
#
# Non-empty Query with F1 = 0:
#     genuine observation
#     -> retain
# ============================================================

blank_query_mask = (
    df["Query"].isna()
    |
    (
        df["Query"]
        .fillna("")
        .astype(str)
        .str.strip()
        == ""
    )
)

blank_rows = int(blank_query_mask.sum())

df = df.loc[~blank_query_mask].copy()


print(
    "Blank-query placeholder rows skipped:",
    blank_rows
)

print(
    "Valid rows retained:",
    len(df)
)

print(
    "Algorithms found:",
    df["Algorithm"].nunique()
)

print(
    "Concepts found:",
    df["ConceptRow"].nunique()
)


# ============================================================
# GLOBAL VALIDATION
# ============================================================

if len(df) != EXPECTED_VALID_ROWS:

    raise ValueError(
        "Expected "
        + str(EXPECTED_VALID_ROWS)
        + " valid rows, but found "
        + str(len(df))
        + "."
    )


if blank_rows != EXPECTED_BLANK_ROWS:

    raise ValueError(
        "Expected "
        + str(EXPECTED_BLANK_ROWS)
        + " blank-query placeholders, but found "
        + str(blank_rows)
        + "."
    )


if df["Algorithm"].nunique() != EXPECTED_ALGORITHMS:

    raise ValueError(
        "Expected "
        + str(EXPECTED_ALGORITHMS)
        + " algorithms."
    )


if df["ConceptRow"].nunique() != EXPECTED_CONCEPTS:

    raise ValueError(
        "Expected "
        + str(EXPECTED_CONCEPTS)
        + " concepts."
    )


# ============================================================
# ALGORITHM VALIDATION
# ============================================================

available_algorithms = set(
    df["Algorithm"].unique()
)

missing_algorithms = [
    algorithm
    for algorithm in ALGORITHM_ORDER
    if algorithm not in available_algorithms
]

if missing_algorithms:

    raise ValueError(
        "These expected algorithms were not found: "
        + ", ".join(missing_algorithms)
    )


# ============================================================
# VALID VARIANTS PER ALGORITHM
# ============================================================

valid_rows_per_algorithm = (
    df.groupby("Algorithm")
    .size()
)

print()
print("Valid query variants per algorithm:")

for algorithm in ALGORITHM_ORDER:

    count = int(
        valid_rows_per_algorithm.loc[
            algorithm
        ]
    )

    print(
        algorithm,
        ":",
        count
    )

    if count != EXPECTED_VALID_VARIANTS_PER_ALGORITHM:

        raise ValueError(
            algorithm
            + " contains "
            + str(count)
            + " valid variants instead of "
            + str(EXPECTED_VALID_VARIANTS_PER_ALGORITHM)
            + "."
        )


# ============================================================
# VALIDATE mc
#
# mc = number of available/non-empty variants for concept c
#
# Expected:
#
# 2 variants = 145 concepts
# 3 variants = 716 concepts
# 4 variants = 668 concepts
# 5 variants = 421 concepts
# 6 variants = 491 concepts
# ============================================================

variant_counts = (
    df.groupby(
        [
            "Algorithm",
            "ConceptRow"
        ]
    )["VariantNo"]
    .nunique()
)

bad_variant_counts = (
    variant_counts[
        (variant_counts < 2)
        |
        (variant_counts > 6)
    ]
)

if not bad_variant_counts.empty:

    print()
    print(
        "Invalid algorithm/concept clusters:"
    )

    print(
        bad_variant_counts.head(20)
    )

    raise ValueError(
        "Some algorithm/concept groups contain "
        "an invalid number of available variants."
    )


print()
print(
    "Validation passed: every algorithm/concept "
    "contains between 2 and 6 valid variants."
)


# ============================================================
# PRINT VARIANT-COUNT DISTRIBUTION
# ============================================================

print()
print("Variant-count distribution per algorithm:")

for algorithm in ALGORITHM_ORDER:

    counts = variant_counts.loc[
        algorithm
    ]

    print()
    print(algorithm)

    for m in range(2, 7):

        number_of_concepts = int(
            (counts == m).sum()
        )

        print(
            "  "
            + str(m)
            + " variants: "
            + str(number_of_concepts)
            + " concepts"
        )


# ============================================================
# CONCEPT-LEVEL AGGREGATION
# ============================================================
#
# For algorithm A and concept c:
#
# Xbar_A,c =
#
#       1
#     -----  SUM X_A,c,v
#      mc
#
# where the sum contains ONLY the mc available variants.
#
# Figure 1:
# X = RetrievedCount
#
# Figure 2:
# X = F1
# ============================================================

concept_df = (
    df.groupby(
        [
            "Algorithm",
            "ConceptRow"
        ],
        as_index=False
    )
    .agg(
        CandidateSetMean=(
            "RetrievedCount",
            "mean"
        ),
        F1Mean=(
            "F1",
            "mean"
        ),
        AvailableVariants=(
            "VariantNo",
            "nunique"
        )
    )
)


# ============================================================
# VALIDATE CONCEPT-LEVEL DATA
# ============================================================

concept_counts = (
    concept_df
    .groupby("Algorithm")
    .size()
)

print()
print("Concept-level observations:")

for algorithm in ALGORITHM_ORDER:

    count = int(
        concept_counts.loc[
            algorithm
        ]
    )

    print(
        algorithm,
        ":",
        count
    )

    if count != EXPECTED_CONCEPTS:

        raise ValueError(
            algorithm
            + " contains "
            + str(count)
            + " concept-level observations instead of "
            + str(EXPECTED_CONCEPTS)
            + "."
        )


print()
print(
    "Concept-level validation passed: "
    "2,441 observations per algorithm."
)


# ============================================================
# MATPLOTLIB PUBLICATION SETTINGS
# ============================================================

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 10,
    "axes.labelsize": 11,
    "xtick.labelsize": 8.5,
    "ytick.labelsize": 9,
    "figure.dpi": 120,
    "savefig.dpi": 600,
    "pdf.fonttype": 42,
    "ps.fonttype": 42
})


# ============================================================
# HELPER FUNCTION
#
# Highlight RomanMetaphone box, whiskers, caps and median
# in red.
#
# All baseline algorithms remain black.
# ============================================================

def highlight_roman_metaphone(bp):

    roman_index = (
        ALGORITHM_ORDER.index(
            "RomanMetaphone"
        )
    )

    # --------------------------------------------------------
    # Box
    # --------------------------------------------------------

    bp["boxes"][roman_index].set_color(
        "red"
    )

    bp["boxes"][roman_index].set_linewidth(
        1.5
    )


    # --------------------------------------------------------
    # Two whiskers per box
    # --------------------------------------------------------

    whisker_start = (
        roman_index * 2
    )

    bp["whiskers"][
        whisker_start
    ].set_color(
        "red"
    )

    bp["whiskers"][
        whisker_start + 1
    ].set_color(
        "red"
    )

    bp["whiskers"][
        whisker_start
    ].set_linewidth(
        1.5
    )

    bp["whiskers"][
        whisker_start + 1
    ].set_linewidth(
        1.5
    )


    # --------------------------------------------------------
    # Two caps per box
    # --------------------------------------------------------

    bp["caps"][
        whisker_start
    ].set_color(
        "red"
    )

    bp["caps"][
        whisker_start + 1
    ].set_color(
        "red"
    )

    bp["caps"][
        whisker_start
    ].set_linewidth(
        1.5
    )

    bp["caps"][
        whisker_start + 1
    ].set_linewidth(
        1.5
    )


    # --------------------------------------------------------
    # Median
    # --------------------------------------------------------

    bp["medians"][
        roman_index
    ].set_color(
        "red"
    )

    bp["medians"][
        roman_index
    ].set_linewidth(
        1.8
    )


# ============================================================
# FIGURE 1
# CONCEPT-LEVEL CANDIDATE-SET SIZE
# ============================================================

candidate_data = []

for algorithm in ALGORITHM_ORDER:

    values = (
        concept_df.loc[
            concept_df["Algorithm"]
            == algorithm,
            "CandidateSetMean"
        ]
        .to_numpy()
    )

    # --------------------------------------------------------
    # log10(mean candidate-set size + 1)
    #
    # +1 preserves zero-result observations.
    # --------------------------------------------------------

    transformed = np.log10(
        values + 1.0
    )

    candidate_data.append(
        transformed
    )


fig, ax = plt.subplots(
    figsize=(10.5, 5.8)
)


bp = ax.boxplot(
    candidate_data,

    tick_labels=[
        DISPLAY_NAMES[a]
        for a in ALGORITHM_ORDER
    ],

    widths=0.60,

    patch_artist=False,

    showfliers=False,

    whis=1.5,

    medianprops={
        "linewidth": 1.5,
        "color": "black"
    },

    boxprops={
        "linewidth": 1.0,
        "color": "black"
    },

    whiskerprops={
        "linewidth": 1.0,
        "color": "black"
    },

    capprops={
        "linewidth": 1.0,
        "color": "black"
    }
)


# ------------------------------------------------------------
# Highlight proposed RomanMetaphone method
# ------------------------------------------------------------

highlight_roman_metaphone(
    bp
)


# ------------------------------------------------------------
# Horizontal breathing room.
#
# Box positions are 1 ... 10.
# Extra space prevents the first and final boxes from appearing
# against or outside the plot boundary.
# ------------------------------------------------------------

ax.set_xlim(
    0.25,
    len(ALGORITHM_ORDER) + 0.75
)


ax.set_ylabel(
    r"Candidate-set size: "
    r"$\log_{10}(\bar{R}_{A,c}+1)$"
)

ax.set_xlabel(
    "Phonetic Algorithms"
)


ax.grid(
    axis="y",
    linestyle="--",
    linewidth=0.5,
    alpha=0.5
)

ax.set_axisbelow(
    True
)


# ------------------------------------------------------------
# Explicit margins for labels and plot boundaries
# ------------------------------------------------------------

fig.subplots_adjust(
    bottom=0.18,
    right=0.97,
    left=0.09,
    top=0.97
)


candidate_pdf = (
    OUTPUT_DIR
    /
    "Figure_CandidateSet_Boxplot.pdf"
)

candidate_png = (
    OUTPUT_DIR
    /
    "Figure_CandidateSet_Boxplot_600dpi.png"
)


fig.savefig(
    candidate_pdf,
    bbox_inches="tight"
)

fig.savefig(
    candidate_png,
    dpi=600,
    bbox_inches="tight"
)


plt.show()

plt.close(
    fig
)


print()
print(
    "Candidate-set figure saved:"
)

print(
    candidate_pdf
)

print(
    candidate_png
)


# ============================================================
# FIGURE 2
# CONCEPT-LEVEL F1
# ============================================================

f1_data = []

for algorithm in ALGORITHM_ORDER:

    values = (
        concept_df.loc[
            concept_df["Algorithm"]
            == algorithm,
            "F1Mean"
        ]
        .to_numpy()
    )

    f1_data.append(
        values
    )


fig, ax = plt.subplots(
    figsize=(10.5, 5.8)
)


bp = ax.boxplot(
    f1_data,

    tick_labels=[
        DISPLAY_NAMES[a]
        for a in ALGORITHM_ORDER
    ],

    widths=0.55,

    patch_artist=False,

    showfliers=False,

    whis=1.5,

    medianprops={
        "linewidth": 1.5,
        "color": "black"
    },

    boxprops={
        "linewidth": 1.0,
        "color": "black"
    },

    whiskerprops={
        "linewidth": 1.0,
        "color": "black"
    },

    capprops={
        "linewidth": 1.0,
        "color": "black"
    }
)


# ------------------------------------------------------------
# Highlight proposed RomanMetaphone method
# ------------------------------------------------------------

highlight_roman_metaphone(
    bp
)


# ------------------------------------------------------------
# Horizontal breathing room.
#
# Prevent first/last boxes from touching the graph boundary.
# ------------------------------------------------------------

ax.set_xlim(
    0.25,
    len(ALGORITHM_ORDER) + 0.75
)


ax.set_ylabel(
    "Concept-level F1"
)

ax.set_xlabel(
    "Phonetic Algorithms"
)


# ------------------------------------------------------------
# F1 theoretical range is 0 ... 1.
#
# The 1.02 upper plotting limit is visual padding only.
# Displayed ticks remain 0.0 ... 1.0.
#
# This prevents whiskers at F1 = 1.0 from visually merging
# with or crossing the upper graph boundary.
# ------------------------------------------------------------

ax.set_ylim(
    0.0,
    1.02
)

ax.set_yticks(
    np.arange(
        0.0,
        1.01,
        0.1
    )
)


ax.grid(
    axis="y",
    linestyle="--",
    linewidth=0.5,
    alpha=0.5
)

ax.set_axisbelow(
    True
)


# ------------------------------------------------------------
# Extra space for multi-line labels and graph boundaries
# ------------------------------------------------------------

fig.subplots_adjust(
    bottom=0.18,
    right=0.97,
    left=0.09,
    top=0.97
)


f1_pdf = (
    OUTPUT_DIR
    /
    "Figure_F1_Boxplot.pdf"
)

f1_png = (
    OUTPUT_DIR
    /
    "Figure_F1_Boxplot_600dpi.png"
)


fig.savefig(
    f1_pdf,
    bbox_inches="tight"
)

fig.savefig(
    f1_png,
    dpi=600,
    bbox_inches="tight"
)


plt.show()

plt.close(
    fig
)


print()
print(
    "F1 figure saved:"
)

print(
    f1_pdf
)

print(
    f1_png
)


print()
print("DONE.")

In [ ]:
# ============================================================
# FIGURE 3
# MACRO PRECISION–RECALL SCATTER
#
# Corrected evaluation basis:
#
# NC = 2,441 concepts
# NT = 10,161 valid query variants per algorithm
#
# Empty Query cells are unavailable spelling variants and
# are excluded completely.
#
# Genuine non-empty queries with Precision = 0 and/or
# Recall = 0 remain valid observations.
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path


# ============================================================
# CONFIGURATION
# ============================================================

INPUT_FILE = (
    r"D:\BORUE"
    r"\PhoneticEvaluation.xlsx"
)

OUTPUT_DIR = Path(
    r"D:\BORUE\PaperFigures10Nov"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


ALGORITHM_ORDER = [
    "Soundex",
    "Refined Soundex",
    "Cologne",
    "NYSIIS",
    "Metaphone",
    "Double Metaphone",
    "Caverphone1",
    "Caverphone2",
    "UrduPhone",
    "RomanMetaphone"
]


# ============================================================
# EXPECTED VALUES
# ============================================================

EXPECTED_PHYSICAL_ROWS = 146460
EXPECTED_BLANK_ROWS = 44850
EXPECTED_VALID_ROWS = 101610

EXPECTED_ALGORITHMS = 10
EXPECTED_VALID_TRIALS_PER_ALGORITHM = 10161


# ============================================================
# LOAD REQUIRED DATA
# ============================================================

print("Reading:", INPUT_FILE)

df = pd.read_excel(
    INPUT_FILE,
    engine="openpyxl",
    usecols=[
        "Algorithm",
        "ConceptRow",
        "VariantNo",
        "Query",
        "Precision",
        "Recall"
    ]
)

print()
print("Physical rows read:", len(df))


# ============================================================
# REMOVE BLANK QUERY PLACEHOLDERS
# ============================================================
#
# Empty Query:
#     unavailable spelling variant
#     -> exclude
#
# Non-empty Query with Precision = 0 or Recall = 0:
#     genuine evaluation trial
#     -> retain
# ============================================================

blank_query_mask = (
    df["Query"].isna()
    |
    (
        df["Query"]
        .fillna("")
        .astype(str)
        .str.strip()
        == ""
    )
)

blank_rows = int(
    blank_query_mask.sum()
)

df = (
    df.loc[~blank_query_mask]
      .copy()
)


print(
    "Blank-query placeholder rows skipped:",
    blank_rows
)

print(
    "Valid rows retained:",
    len(df)
)

print(
    "Algorithms found:",
    df["Algorithm"].nunique()
)


# ============================================================
# VALIDATION
# ============================================================

if blank_rows != EXPECTED_BLANK_ROWS:

    raise ValueError(
        "Expected "
        + str(EXPECTED_BLANK_ROWS)
        + " blank-query placeholders, but found "
        + str(blank_rows)
        + "."
    )


if len(df) != EXPECTED_VALID_ROWS:

    raise ValueError(
        "Expected "
        + str(EXPECTED_VALID_ROWS)
        + " valid rows, but found "
        + str(len(df))
        + "."
    )


if df["Algorithm"].nunique() != EXPECTED_ALGORITHMS:

    raise ValueError(
        "Expected "
        + str(EXPECTED_ALGORITHMS)
        + " algorithms."
    )


missing_algorithms = [
    algorithm
    for algorithm in ALGORITHM_ORDER
    if algorithm not in set(
        df["Algorithm"].unique()
    )
]

if missing_algorithms:

    raise ValueError(
        "These expected algorithms were not found: "
        + ", ".join(missing_algorithms)
    )


# ============================================================
# CALCULATE MACRO PRECISION AND MACRO RECALL
#
# Macro measures are arithmetic means across NT = 10,161
# VALID query-variant trials per algorithm.
#
# Blank structural variant slots have already been excluded.
# ============================================================

pr_summary = (
    df.groupby("Algorithm")
      .agg(
          MacroPrecision=(
              "Precision",
              "mean"
          ),
          MacroRecall=(
              "Recall",
              "mean"
          ),
          Trials=(
              "Precision",
              "size"
          )
      )
      .reindex(
          ALGORITHM_ORDER
      )
      .reset_index()
)


# ============================================================
# VALIDATE NT = 10,161 PER ALGORITHM
# ============================================================

for _, row in pr_summary.iterrows():

    algorithm = row["Algorithm"]
    trials = int(row["Trials"])

    if trials != EXPECTED_VALID_TRIALS_PER_ALGORITHM:

        raise ValueError(
            algorithm
            + " contains "
            + str(trials)
            + " valid trials instead of "
            + str(EXPECTED_VALID_TRIALS_PER_ALGORITHM)
            + "."
        )


print()
print(
    "Validation passed: every algorithm contains "
    "10,161 valid query-variant trials."
)

print()
print("Precision-Recall values:")

print(
    pr_summary.to_string(
        index=False
    )
)


# ============================================================
# MATPLOTLIB PUBLICATION SETTINGS
# ============================================================

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 10,
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "savefig.dpi": 600,
    "pdf.fonttype": 42,
    "ps.fonttype": 42
})


# ============================================================
# PLOT
# ============================================================

fig, ax = plt.subplots(
    figsize=(8.2, 6.2)
)


# ============================================================
# PLOT BASELINE METHODS
#
# All baseline algorithms are black.
# ============================================================

baseline_df = (
    pr_summary[
        pr_summary["Algorithm"]
        != "RomanMetaphone"
    ]
)


ax.scatter(
    baseline_df["MacroRecall"],
    baseline_df["MacroPrecision"],
    s=65,
    color="black",
    zorder=3,
    label="Baseline methods"
)


# ============================================================
# PLOT ROMANMETAPHONE
#
# Proposed method is highlighted in red.
# ============================================================

roman_df = (
    pr_summary[
        pr_summary["Algorithm"]
        == "RomanMetaphone"
    ]
)


ax.scatter(
    roman_df["MacroRecall"],
    roman_df["MacroPrecision"],
    s=85,
    color="red",
    zorder=4,
    label="RomanMetaphone"
)


# ============================================================
# LABEL OFFSETS
#
# Offsets affect text placement only.
# They do NOT alter the actual data-point positions.
# ============================================================

label_offsets = {
    "Soundex": (6, 5),
    "Refined Soundex": (6, 5),
    "Cologne": (6, -12),
    "NYSIIS": (6, -12),
    "Metaphone": (6, 5),
    "Double Metaphone": (6, -13),
    "Caverphone1": (6, 5),
    "Caverphone2": (6, -12),
    "UrduPhone": (6, 5),
    "RomanMetaphone": (6, 6)
}


# ============================================================
# ANNOTATE EVERY DATA POINT
#
# IMPORTANT:
# ax.annotate() MUST remain inside this loop.
#
# RomanMetaphone is displayed as "RuMetA" in the figure. Changed later to RASOM
# The underlying algorithm name remains RomanMetaphone.
# ============================================================

for _, row in pr_summary.iterrows():

    algorithm = row["Algorithm"]

    dx, dy = label_offsets.get(
        algorithm,
        (5, 5)
    )

    if algorithm == "RomanMetaphone":

        label_color = "red"
        label_weight = "bold"
        display_label = "RASOM"

    else:

        label_color = "black"
        label_weight = "normal"
        display_label = algorithm

    ax.annotate(
        display_label,
        (
            row["MacroRecall"],
            row["MacroPrecision"]
        ),
        xytext=(
            dx,
            dy
        ),
        textcoords="offset points",
        fontsize=8.5,
        color=label_color,
        fontweight=label_weight
    )


# ============================================================
# AXES
# ============================================================

ax.set_xlabel(
    "Macro Recall"
)

ax.set_ylabel(
    "Macro Precision"
)


ax.grid(
    linestyle="--",
    linewidth=0.5,
    alpha=0.5
)

ax.set_axisbelow(
    True
)


# ============================================================
# AXIS LIMITS
#
# Add visual breathing room around observed values.
#
# These limits DO NOT alter the plotted data.
# ============================================================

x_min = (
    pr_summary["MacroRecall"].min()
)

x_max = (
    pr_summary["MacroRecall"].max()
)

y_min = (
    pr_summary["MacroPrecision"].min()
)

y_max = (
    pr_summary["MacroPrecision"].max()
)


ax.set_xlim(
    max(
        0.0,
        x_min - 0.035
    ),
    min(
        1.0,
        x_max + 0.055
    )
)


ax.set_ylim(
    max(
        0.0,
        y_min - 0.035
    ),
    min(
        1.0,
        y_max + 0.055
    )
)


# ============================================================
# LAYOUT
# ============================================================

fig.tight_layout()


# ============================================================
# SAVE
# ============================================================

pdf_file = (
    OUTPUT_DIR
    /
    "Figure_PrecisionRecall_Scatter.pdf"
)

png_file = (
    OUTPUT_DIR
    /
    "Figure_PrecisionRecall_Scatter_600dpi.png"
)


fig.savefig(
    pdf_file,
    bbox_inches="tight"
)

fig.savefig(
    png_file,
    dpi=600,
    bbox_inches="tight"
)


plt.show()

plt.close(
    fig
)


print()
print("Figure 3 saved:")

print(pdf_file)
print(png_file)

In [ ]:
# ============================================================
# FIGURE 4
# CONCEPT-LEVEL HIT RATE
#
# Corrected evaluation basis:
#
# NC = 2,441 concepts
# NT = 10,161 valid query variants per algorithm
# mc = 2 to 6 available spelling variants per concept
#
# Empty Query cells are structural placeholders and are
# excluded completely.
#
# A concept is a hit if ANY of its available variants
# produces Hit = 1.
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path


# ============================================================
# CONFIGURATION
# ============================================================

INPUT_FILE = (
    r"D:\BORUE"
    r"\PhoneticEvaluation.xlsx"
)

OUTPUT_DIR = Path(
    r"D:\BORUE\PaperFigures10Nov"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


ALGORITHM_ORDER = [
    "Soundex",
    "Refined Soundex",
    "Cologne",
    "NYSIIS",
    "Metaphone",
    "Double Metaphone",
    "Caverphone1",
    "Caverphone2",
    "UrduPhone",
    "RomanMetaphone"
]


DISPLAY_NAMES = {
    "Soundex": "Soundex",
    "Refined Soundex": "Refined\nSoundex",
    "Cologne": "Cologne",
    "NYSIIS": "NYSIIS",
    "Metaphone": "Metaphone",
    "Double Metaphone": "Double\nMetaphone",
    "Caverphone1": "Caverphone1",
    "Caverphone2": "Caverphone2",
    "UrduPhone": "UrduPhone",
    "RomanMetaphone": "RASOM"
}


# ============================================================
# EXPECTED DATASET VALUES
# ============================================================

EXPECTED_PHYSICAL_ROWS = 146460
EXPECTED_BLANK_ROWS = 44850
EXPECTED_VALID_ROWS = 101610

EXPECTED_ALGORITHMS = 10
EXPECTED_CONCEPTS = 2441
EXPECTED_VALID_TRIALS_PER_ALGORITHM = 10161


# ============================================================
# LOAD DATA
# ============================================================

print("Reading:", INPUT_FILE)

df = pd.read_excel(
    INPUT_FILE,
    engine="openpyxl",
    usecols=[
        "Algorithm",
        "ConceptRow",
        "VariantNo",
        "Query",
        "Hit"
    ]
)

print()
print("Physical rows read:", len(df))


# ============================================================
# REMOVE BLANK-QUERY PLACEHOLDERS
# ============================================================
#
# Empty Query:
#     unavailable spelling variant
#     -> exclude
#
# Non-empty Query with Hit = 0:
#     genuine retrieval failure
#     -> retain
# ============================================================

blank_query_mask = (
    df["Query"].isna()
    |
    (
        df["Query"]
        .fillna("")
        .astype(str)
        .str.strip()
        == ""
    )
)

blank_rows = int(
    blank_query_mask.sum()
)

df = (
    df.loc[~blank_query_mask]
      .copy()
)


print(
    "Blank-query placeholder rows skipped:",
    blank_rows
)

print(
    "Valid rows retained:",
    len(df)
)

print(
    "Algorithms found:",
    df["Algorithm"].nunique()
)

print(
    "Concepts found:",
    df["ConceptRow"].nunique()
)


# ============================================================
# GLOBAL VALIDATION
# ============================================================

if blank_rows != EXPECTED_BLANK_ROWS:

    raise ValueError(
        "Expected "
        + str(EXPECTED_BLANK_ROWS)
        + " blank-query placeholders, but found "
        + str(blank_rows)
        + "."
    )


if len(df) != EXPECTED_VALID_ROWS:

    raise ValueError(
        "Expected "
        + str(EXPECTED_VALID_ROWS)
        + " valid rows, but found "
        + str(len(df))
        + "."
    )


if df["Algorithm"].nunique() != EXPECTED_ALGORITHMS:

    raise ValueError(
        "Expected "
        + str(EXPECTED_ALGORITHMS)
        + " algorithms."
    )


if df["ConceptRow"].nunique() != EXPECTED_CONCEPTS:

    raise ValueError(
        "Expected "
        + str(EXPECTED_CONCEPTS)
        + " concepts."
    )


missing_algorithms = [
    algorithm
    for algorithm in ALGORITHM_ORDER
    if algorithm not in set(
        df["Algorithm"].unique()
    )
]

if missing_algorithms:

    raise ValueError(
        "These expected algorithms were not found: "
        + ", ".join(missing_algorithms)
    )


# ============================================================
# VALIDATE NT = 10,161 PER ALGORITHM
# ============================================================

valid_rows_per_algorithm = (
    df.groupby("Algorithm")
      .size()
)


print()
print("Valid query variants per algorithm:")


for algorithm in ALGORITHM_ORDER:

    count = int(
        valid_rows_per_algorithm.loc[
            algorithm
        ]
    )

    print(
        algorithm,
        ":",
        count
    )

    if count != EXPECTED_VALID_TRIALS_PER_ALGORITHM:

        raise ValueError(
            algorithm
            + " contains "
            + str(count)
            + " valid variants instead of "
            + str(EXPECTED_VALID_TRIALS_PER_ALGORITHM)
            + "."
        )


# ============================================================
# VALIDATE mc
#
# mc = number of available variants for concept c
#
# Every concept must contain between 2 and 6 valid variants.
# ============================================================

variant_counts = (
    df.groupby(
        [
            "Algorithm",
            "ConceptRow"
        ]
    )["VariantNo"]
    .nunique()
)


bad_variant_counts = (
    variant_counts[
        (variant_counts < 2)
        |
        (variant_counts > 6)
    ]
)


if not bad_variant_counts.empty:

    print()
    print(
        "Invalid algorithm/concept clusters:"
    )

    print(
        bad_variant_counts.head(20)
    )

    raise ValueError(
        "Some algorithm/concept groups contain "
        "an invalid number of available variants."
    )


print()
print(
    "Validation passed: every algorithm/concept "
    "contains between 2 and 6 valid variants."
)


# ============================================================
# CONCEPT-LEVEL HIT
#
# ConceptHit = 1 if ANY of the mc available variants
# has Hit = 1.
#
# max(Hit) therefore implements the same concept-level
# definition used in the McNemar analysis.
# ============================================================

concept_hit = (
    df.groupby(
        [
            "Algorithm",
            "ConceptRow"
        ],
        as_index=False
    )
    .agg(
        ConceptHit=(
            "Hit",
            "max"
        ),
        AvailableVariants=(
            "VariantNo",
            "nunique"
        )
    )
)


# ============================================================
# VALIDATE NC = 2,441 CONCEPTS PER ALGORITHM
# ============================================================

concept_counts = (
    concept_hit
    .groupby("Algorithm")
    .size()
)


for algorithm in ALGORITHM_ORDER:

    count = int(
        concept_counts.loc[
            algorithm
        ]
    )

    if count != EXPECTED_CONCEPTS:

        raise ValueError(
            algorithm
            + " contains "
            + str(count)
            + " concepts instead of "
            + str(EXPECTED_CONCEPTS)
            + "."
        )


print(
    "Concept-level validation passed: "
    "2,441 concepts per algorithm."
)


# ============================================================
# CALCULATE CONCEPT-LEVEL HIT RATE
# ============================================================

hit_summary = (
    concept_hit
    .groupby("Algorithm")
    .agg(
        ConceptHitRate=(
            "ConceptHit",
            "mean"
        ),
        SuccessfulConcepts=(
            "ConceptHit",
            "sum"
        ),
        TotalConcepts=(
            "ConceptHit",
            "size"
        )
    )
    .reindex(
        ALGORITHM_ORDER
    )
    .reset_index()
)


hit_summary[
    "ConceptHitPercent"
] = (
    hit_summary[
        "ConceptHitRate"
    ] * 100.0
)


print()
print("Concept-level Hit Rates:")

print(
    hit_summary.to_string(
        index=False
    )
)


# ============================================================
# MATPLOTLIB PUBLICATION SETTINGS
# ============================================================

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 10,
    "axes.labelsize": 11,
    "xtick.labelsize": 8.5,
    "ytick.labelsize": 9,
    "savefig.dpi": 600,
    "pdf.fonttype": 42,
    "ps.fonttype": 42
})


# ============================================================
# PLOT
# ============================================================

fig, ax = plt.subplots(
    figsize=(10.5, 5.8)
)


# ============================================================
# COLORS
#
# Baselines = gray
# RomanMetaphone = red
# ============================================================

bar_colors = [
    "seagreen"
    if algorithm == "RomanMetaphone"
    else "cornflowerblue"
    for algorithm
    in hit_summary["Algorithm"]
]


bars = ax.bar(
    range(
        len(hit_summary)
    ),
    hit_summary[
        "ConceptHitPercent"
    ],
    width=0.65,
    color=bar_colors
)


# ============================================================
# X AXIS
# ============================================================

ax.set_xticks(
    range(
        len(hit_summary)
    )
)


ax.set_xticklabels(
    [
        DISPLAY_NAMES[a]
        for a
        in hit_summary["Algorithm"]
    ]
)


ax.set_xlabel(
    "Phonetic Algorithms"
)

ax.set_ylabel(
    "Concept-level Hit Rate (%)"
)


# ============================================================
# Y AXIS
#
# Hit Rate has a natural 0–100% scale.
#
# We use 102 only as visual headroom so labels or a bar
# reaching 100% cannot collide with the upper border.
#
# Displayed ticks remain 0–100%.
# ============================================================

ax.set_ylim(
    0,
    102
)


ax.set_yticks(
    range(
        0,
        101,
        10
    )
)


# ============================================================
# GRID
# ============================================================

ax.grid(
    axis="y",
    linestyle="--",
    linewidth=0.5,
    alpha=0.5
)

ax.set_axisbelow(
    True
)


# ============================================================
# DISPLAY EXACT PERCENTAGE ABOVE EACH BAR
# ============================================================

for bar, value, algorithm in zip(
    bars,
    hit_summary["ConceptHitPercent"],
    hit_summary["Algorithm"]
):

    text_color = (
        "red"
        if algorithm == "RomanMetaphone"
        else "black"
    )

    text_weight = (
        "bold"
        if algorithm == "RomanMetaphone"
        else "normal"
    )

    ax.text(
        bar.get_x()
        + bar.get_width() / 2,

        bar.get_height()
        + 1.0,

        f"{value:.1f}%",

        ha="center",
        va="bottom",
        fontsize=8.5,
        color=text_color,
        fontweight=text_weight
    )


# ============================================================
# EXTRA HORIZONTAL BREATHING ROOM
# ============================================================

ax.set_xlim(
    -0.75,
    len(hit_summary) - 0.25
)


# ============================================================
# LAYOUT
# ============================================================

fig.subplots_adjust(
    bottom=0.18,
    right=0.97,
    left=0.09,
    top=0.97
)


# ============================================================
# SAVE
# ============================================================

pdf_file = (
    OUTPUT_DIR
    /
    "Figure_ConceptHitRate_Bar.pdf"
)

png_file = (
    OUTPUT_DIR
    /
    "Figure_ConceptHitRate_Bar_600dpi.png"
)


fig.savefig(
    pdf_file,
    bbox_inches="tight"
)


fig.savefig(
    png_file,
    dpi=600,
    bbox_inches="tight"
)


plt.show()

plt.close(
    fig
)


print()
print("Figure 4 saved:")

print(pdf_file)
print(png_file)

In [ ]:
#...............just to show sample colors
import matplotlib.pyplot as plt

colors = ['steelblue', 'seagreen', 'crimson', 'darkorchid', 'goldenrod','cornflowerblue']
values = [12, 19, 7, 14, 9,10]
labels = ['Blue', 'Green', 'Red', 'Purple', 'Gold','cornflowerblue']

plt.bar(labels, values, color=colors)
plt.title('Custom Named Colors Example')
plt.show()
